# Day 5 Laboratory Exercise: What The Answer Does Not Say

This morning's walkthrough built an agent out of five parts and then watched a
library build the same thing. It showed that a model with no tool answers a
database question from memory and gets some of it wrong, that a tool is a JSON
description plus a function you wrote, that the loop is twenty-three lines, and
that a broken query gets repaired because the tool hands the error back as text.

This afternoon you are going to check whether the agent is any good, which is a
different question from whether it works. Working is what you saw this morning.
Being good is a claim about questions it has not been asked yet, and the only
way to support that claim is to ask several and score them against answers you
already know.

So you will write the answers first, in SQL, by hand. Then you will build the
agent, ask it the same questions, and mark it.

It will score full marks. The exercise is what you find after that.

## How To Work Through This

There are eight tasks. Each states a goal, gives you a cell marked
`# YOUR CODE HERE` naming the variables the check expects, and follows it with a
check cell that either confirms your answer or tells you what it wanted.

## A Word About Checks Today

Every check on the first four days compared a number against a number, and it
could, because the same input gave the same output. Today a language model sits
in the middle, and there is no promise anywhere that it says the same words
twice.

So no check below compares the model's words to anything. The checks fall into
two kinds. Where a fact comes from the database, such as the number of films or
the money the Action rentals took, the check asserts it exactly, because that
number is true whatever any model says about it. Where a quantity comes from the
model, such as how many calls a question took, the check asserts a relationship
rather than a value, because the relationship is the thing that carries the
lesson.

Your agent's sentences will not match the ones printed in the solution word for
word, and that is not a failure. What has to match is the number inside them.

In [ ]:
import json
import os
import re
import sqlite3
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

DATA_DIR = Path.cwd().parent / "data"
DATABASE = DATA_DIR / "sqlite-sakila.db"

RANDOM_STATE = 42

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://vllm.finki.ukim.mk/v1")
MODELS = ["qwen3.8-27b", "big-pickle", "mimo-v2.5-free"]


def check(condition, success, failure):
    """Report whether a task was completed correctly."""
    print(success if condition else failure)


def read_api_key():
    """Find the key in the two places that are outside this repository."""
    from_environment = os.environ.get("OPENAI_API_KEY", "").strip()
    if from_environment:
        return from_environment, "the OPENAI_API_KEY environment variable"

    from_file = Path.home() / ".mltp2026-key"
    if from_file.exists() and from_file.read_text().strip():
        return from_file.read_text().strip(), str(from_file)

    raise RuntimeError(
        "No key was found, so nothing in this notebook can run. Write yours "
        "into the file ~/.mltp2026-key, or set OPENAI_API_KEY in the "
        "environment, and run this cell again.")


def choose_model(client, names):
    """Return the first alias that answers."""
    for name in names:
        try:
            client.chat.completions.create(
                model=name, messages=[{"role": "user", "content": "ping"}],
                max_tokens=1, temperature=0)
            return name
        except Exception as error:
            print(f"  {name} did not answer: {type(error).__name__}")
    raise RuntimeError(
        "None of the models answered. Check the endpoint and the key before "
        "going any further.")


api_key, found_in = read_api_key()
client = OpenAI(base_url=BASE_URL, api_key=api_key, max_retries=5, timeout=600)
MODEL = choose_model(client, MODELS)

# Read-only, and shared with a worker thread by Task 3's machinery. Nothing
# here runs two queries at once, which is what makes that safe.
connection = sqlite3.connect(f"file:{DATABASE}?mode=ro", uri=True,
                             check_same_thread=False)
TABLES = [name for (name,) in connection.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]

print("key read from:", found_in)
print("model:        ", MODEL)
print("run at:       ", time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
print("tables:       ", len(TABLES))

## The Tools You Are Given

The next cell holds two things. Read them, run it, and then leave it alone.

`run_agent` is the loop from Section 3 of the walkthrough, with one addition:
it keeps what every statement returned as well as the statement itself, because
several tasks below need to look at what came back rather than only at what was
asked. It is given to you because you watched it being written this morning and
typing it again teaches nothing.

`numbers_in` pulls the numbers out of a sentence. It is the small piece of
machinery that makes this afternoon possible. You cannot compare a model's
prose against an expected string, but you can pull `77` out of whatever
sentence the model wrapped around it and compare that.

In [ ]:
def run_agent(question, system, tool, tools_description, max_steps=8):
    """Let the model call the tool until it stops asking, and record everything.

    Returns the answer, the statements the model asked for, what each one
    returned, the number of model calls, and the tokens those calls cost.
    """
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    statements, results = [], []
    calls, tokens_in, tokens_out = 0, 0, 0

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL, temperature=0, messages=messages,
            tools=tools_description)
        calls += 1
        tokens_in += response.usage.prompt_tokens
        tokens_out += response.usage.completion_tokens

        reply = response.choices[0].message
        messages.append(reply.model_dump(exclude_none=True))

        if not reply.tool_calls:
            return dict(answer=reply.content, statements=statements,
                        results=results, calls=calls, tokens_in=tokens_in,
                        tokens_out=tokens_out)

        for call in reply.tool_calls:
            query = json.loads(call.function.arguments or "{}").get("query", "")
            outcome = tool(query)
            statements.append(query)
            results.append(outcome)
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": outcome})

    return dict(answer=None, statements=statements, results=results,
                calls=calls, tokens_in=tokens_in, tokens_out=tokens_out)


def numbers_in(text):
    """Every number in a piece of prose, with thousands separators removed."""
    return [float(found.replace(",", ""))
            for found in re.findall(r"\d[\d,]*\.?\d*", text or "")]


def says(text, value, tolerance=0.01):
    """Whether a sentence contains a number close enough to the one wanted."""
    room = max(tolerance, abs(value) * 0.001)
    return any(abs(found - value) <= room for found in numbers_in(text))


print("numbers_in on a sentence the model might write:")
print(" ", numbers_in("The stores earned a total of **$4,375.85** last year."))
print("says(..., 4375.85):",
      says("The stores earned a total of **$4,375.85** last year.", 4375.85))

## Task 1: Write The Answers Down First

Before there is an agent to test, work out what the right answers are.

This is the part everybody skips, and skipping it turns an evaluation into a
demonstration. If you build the agent first and then think of questions, you
will think of questions it handled, because those are the ones in front of you.

Here are five questions. Answer all five in SQL, against `connection`.

1. How many films are in the database?
2. What is the average length of a film, in minutes?
3. How many films mention a robot in their description?
4. How much money did the stores take from renting Action films?
5. How many customers are there?

Store the five answers in a dictionary called `reference`, keyed by `"films"`,
`"avg_length"`, `"robot"`, `"action_revenue"`, and `"customers"`, with a number
for each value.

Two notes on the harder ones. Search a description case-insensitively, because
the descriptions are not written in one case. Money reaches a film through
`payment`, then `rental`, then `inventory`, then `film_category`, then
`category`.

In [ ]:
# YOUR CODE HERE
# Answer the five questions in SQL and store them in `reference`.

In [ ]:
check(
    "reference" in dir()
    and set(reference) == {"films", "avg_length", "robot", "action_revenue",
                           "customers"}
    and reference["films"] == 1000
    and abs(reference["avg_length"] - 115.272) < 0.001
    and reference["robot"] == 77
    and abs(reference["action_revenue"] - 4375.85) < 0.01
    and reference["customers"] == 599,
    "Task 1 complete. 1000 films, an average length of 115.272 minutes, 77 "
    "films mentioning a robot, 4375.85 taken on Action rentals, and 599 "
    "customers. Those five numbers are the only thing in this notebook that "
    "no model can change.",
    "Not right yet. Expected films 1000, avg_length 115.272, robot 77, "
    "action_revenue 4375.85, and customers 599.",
)

## Task 2: Build The Tool, And The Guard That Goes With It

The agent needs one tool. Write it, and write the guard in the same breath,
because the walkthrough showed what an unguarded one does when it is pointed at
a database that permits writing.

Write a function `run_sql(query)` that:

- refuses a query containing a semicolon anywhere except at the very end,
  returning a string that starts with `"refused"`,
- refuses a query that does not begin with `select`, `pragma`, or `with`,
  ignoring case, returning a string that starts with `"refused"`,
- otherwise runs it against `connection`, returning the rows as JSON, and
- returns at most 20 rows, saying how many there were when it cuts.

An error must come back as text beginning with `"SQL error:"` rather than being
raised, because the model reads that text and repairs itself with it.

`pragma` and `with` are on the list because a read-only tool that refuses them
stops the model inspecting the schema, and that is a large part of what it
needs to do.

Then write `TOOLS`, the JSON description of that function, in the shape the
walkthrough used. Call the tool `run_sql` there too.

In [ ]:
# YOUR CODE HERE
# Write `run_sql(query)` with its guard, and the `TOOLS` description of it.

In [ ]:
check(
    "run_sql" in dir() and "TOOLS" in dir()
    and run_sql("SELECT COUNT(*) FROM film") == "[[1000]]"
    and run_sql("DELETE FROM film").startswith("refused")
    and run_sql("SELECT 1; DROP TABLE film").startswith("refused")
    and run_sql("SELECT nope FROM film").startswith("SQL error:")
    and run_sql("PRAGMA table_info(film)").startswith("[")
    and "1000 rows, 20 shown" in run_sql("SELECT title FROM film")
    and TOOLS[0]["function"]["name"] == "run_sql",
    "Task 2 complete. The tool counts films, refuses a DELETE, refuses two "
    "statements at once, hands an error back as text, allows PRAGMA, and cuts "
    "1000 titles down to 20 while saying that it did.",
    "Not right yet. run_sql should return '[[1000]]' for a count of films, a "
    "string starting 'refused' for a DELETE and for two statements, a string "
    "starting 'SQL error:' for an unknown column, JSON for a PRAGMA, and at "
    "most 20 rows with a note saying 1000 rows were found.",
)

## Task 3: Wire It Together And Ask One Question

Write the system message and make one run.

Store `SYSTEM`, a string that tells the model it answers questions about a film
rental database by writing SQL, names the tables in `TABLES`, and asks it to
call `run_sql` and then answer in one sentence. That is the message the
walkthrough used.

Then call `run_agent` with the first question and store the result as
`first_run`. The signature is
`run_agent(question, system, tool, tools_description)`.

In [ ]:
# YOUR CODE HERE
# Store `SYSTEM` and `first_run`.

In [ ]:
check(
    "SYSTEM" in dir() and "first_run" in dir() and "reference" in dir()
    and "run_sql" in SYSTEM
    and all(table in SYSTEM for table in TABLES)
    and first_run["calls"] >= 2
    and len(first_run["statements"]) >= 1
    and says(first_run["answer"], reference["films"]),
    "Task 3 complete. The agent ran a query and its sentence contains 1000, "
    "which is the number you computed in Task 1. Note that it took at least "
    "two calls to answer one question, because one call was spent asking.",
    "Not right yet. SYSTEM should name run_sql and every table, and the run "
    "should take at least two calls, run at least one statement, and produce "
    "a sentence containing 1000.",
)

## Task 4: Ask All Five, And Mark Them

Now the evaluation. Ask the agent all five of Task 1's questions and score each
answer against the reference.

Store:

- `questions`, a dictionary from the five keys to the five questions as you
  would put them to a person,
- `runs`, a dictionary from those keys to what `run_agent` returned, and
- `score`, how many of the five answers contain the right number.

Use `says(answer, expected)` to mark an answer. Do not compare strings.

This makes about a dozen calls and takes a minute or two. Start it and read
ahead.

In [ ]:
# YOUR CODE HERE
# Store `questions`, `runs`, and `score`.

In [ ]:
check(
    "runs" in dir() and "score" in dir() and "questions" in dir()
    and "reference" in dir()
    and set(runs) == set(reference)
    and score == 5,
    "Task 4 complete. Five out of five. On the evidence in front of you the "
    "agent is finished, and you could put that number on a slide. Do not stop "
    "here.",
    "Not right yet. Expected `runs` to hold a result for each of the five "
    "keys and `score` to be 5. If an answer is marked wrong, print it and "
    "read what the agent actually said before changing anything.",
)

## Task 5: Now Count What It Cost

The score says every answer was right. It says nothing at all about what
getting them cost, so measure that separately.

Build a table `costs`, a pandas `DataFrame` indexed by the five keys, with a
column `calls`, a column `tokens_in`, a column `tokens_out`, and a column
`statements` holding how many SQL statements each question needed.

Then store:

- `dearest`, the key of the question that took the most model calls, and
- `ratio`, the most calls any question took divided by the fewest.

In [ ]:
# YOUR CODE HERE
# Store `costs`, `dearest`, and `ratio`.

In [ ]:
check(
    "costs" in dir() and "dearest" in dir() and "ratio" in dir()
    and len(costs) == 5
    and set(costs.columns) >= {"calls", "tokens_in", "tokens_out",
                               "statements"}
    and dearest == "robot"
    and ratio >= 2,
    "Task 5 complete. Every answer was right and one of them cost at least "
    "twice as many calls as the cheapest. The question about robots is the "
    "expensive one, and nothing in its answer told you so.",
    "Not right yet. `costs` needs a row per question and columns calls, "
    "tokens_in, tokens_out, and statements. `dearest` should come out as "
    "'robot' and `ratio` should be at least 2.",
)

## Task 6: Read The Expensive Run

A number told you which question was dear. It cannot tell you why, and the
answer certainly will not. The trajectory will, because `run_agent` kept it.

Print every statement of the `dearest` run beside what it returned, read them
in order, and then store:

- `empty_results`, how many of that run's statements returned no rows at all,
  meaning the result was exactly `"[]"`, and
- `blind_table`, the name of the table those statements were querying.

In [ ]:
# YOUR CODE HERE
# Store `empty_results` and `blind_table`.

In [ ]:
check(
    "empty_results" in dir() and "blind_table" in dir()
    and blind_table == "film_text"
    and empty_results >= 1,
    "Task 6 complete. At least one statement asked film_text and got an empty "
    "list back. Notice what did not happen: no error, no warning, and nothing "
    "in the tool's reply to say that the question was fine and the table was "
    "not.",
    "Not right yet. Look for the statements whose result is exactly '[]'. "
    "`empty_results` should count them and `blind_table` should be the name "
    "of the table they query.",
)

## Task 7: Ask The Database What Happened

Stop asking the model and ask the file. Store:

- `film_text_rows`, the number of rows in `film_text`,
- `film_rows`, the number of rows in `film`, and
- `robot_in_film_text`, the number of rows in `film_text` whose description
  mentions a robot.

In [ ]:
# YOUR CODE HERE
# Store `film_text_rows`, `film_rows`, and `robot_in_film_text`.

In [ ]:
check(
    "film_text_rows" in dir() and "film_rows" in dir()
    and "robot_in_film_text" in dir()
    and film_text_rows == 0
    and film_rows == 1000
    and robot_in_film_text == 0,
    "Task 7 complete. film_text has 0 rows and film has 1000. The table is "
    "real, its columns are real, the query was valid, and the answer was "
    "nothing. A model reading the list of tables has no way to tell that "
    "apart from a table that is full.",
    "Not right yet. Expected film_text to have 0 rows, film to have 1000, and "
    "0 films mentioning a robot according to film_text.",
)

### Why That Table Is Empty

It is a property of this file rather than a fault in it. In MySQL, where Sakila
comes from, `film_text` is filled from `film` by triggers so that a full text
index can be built over it. The port to SQLite creates the table and does not
create the triggers, so it is empty and stays empty. `data/SOURCES.md` records
this.

The agent was not wrong to look there. `film_text` is the obvious place to find
the text of a film, and its name says so. It was wrong to believe the answer.

## Task 8: Fix It, And Find Out What The Fix Costs

The agent was told the names of the tables. It was not told which of them hold
anything, and that is the whole of the missing information.

Give it that. Store:

- `SYSTEM_WITH_COUNTS`, the same system message with the number of rows written
  beside each table name, and
- `fixed`, the result of asking the `dearest` question again with it, and
- `saved`, how many calls that saved.

Then run all five questions again under the new message and store `runs_fixed`
and `score_fixed`.

In [ ]:
# YOUR CODE HERE
# Store `SYSTEM_WITH_COUNTS`, `fixed`, `saved`, `runs_fixed`, and `score_fixed`.

In [ ]:
check(
    "SYSTEM_WITH_COUNTS" in dir() and "fixed" in dir() and "saved" in dir()
    and "runs_fixed" in dir() and "score_fixed" in dir()
    and "SYSTEM" in dir() and "reference" in dir()
    and len(SYSTEM_WITH_COUNTS) > len(SYSTEM)
    and says(fixed["answer"], reference["robot"])
    and saved >= 1
    and score_fixed == 5,
    "Task 8 complete. The same answer, still 77, reached in fewer calls, and "
    "the other four questions are still right. Now look at the row of the "
    "table for the customers question before you read the next cell.",
    "Not right yet. SYSTEM_WITH_COUNTS should be longer than SYSTEM, `fixed` "
    "should still contain 77, `saved` should be at least 1, and score_fixed "
    "should still be 5.",
)

### The Row Worth Looking At

The fix worked on the question it was aimed at. Look at what it did to a
question it was not aimed at.

The customers question needs one number, the count of rows in `customer`, and
the new system message contains that number. So the agent read it off the
prompt and answered without running anything. Its `statements after` is zero.

The answer is right. It is also no longer an answer about the database, because
nothing in that run touched the database. You moved a fact out of the data and
into the prompt, and the agent did what anybody would do with a fact within
reach, which is to use it instead of going to look.

That is the trade this fix makes. It is not an argument against making it. It
is an argument for knowing that you made it, because the next time that number
changes, the database will be right and the prompt will be wrong, and the score
will still be five out of five.

## What You Found

Write this down before you leave, because it is the thing that generalises past
this database.

The agent scored five out of five. On that evidence there is nothing to say
about it except that it works.

One of those five answers was reached after two queries against an empty table
and two more spent working out why, at two and a half times the calls and more
than four times the prompt tokens of the cheapest question. Every part of that is absent
from the sentence the agent produced, which was as fluent, as confident, and as
correct as the other four.

So the answer is not the output of an agent. The trajectory is. An answer can
be checked against a reference, and checking it is necessary, but a correct
answer establishes only that this question, this time, came out right. What
went wrong on the way, what it cost, and whether it would survive the same
question worded differently are all in the transcript, and the transcript is
the part nobody reads.

Three habits follow, and none of them is specific to SQL.

- Write the answers before you build the thing that produces them. Five
  questions with known answers took you ten minutes in Task 1 and are the only
  reason any statement in this notebook can be believed.
- Count the calls. Cost per question varied by a factor of two and a half here,
  it was invisible in every answer, and it is the number that decides whether a
  demonstration can become something people use.
- Read the transcript of anything expensive. The price was the symptom and the
  empty table was the cause, and no amount of staring at the answers would have
  found it.

And one about this database in particular, which is really about all of them. A
schema tells you what tables exist. It does not tell you which of them anybody
filled.

## If You Have Time

These are not marked and there is no check cell for them.

1. Ask the fifth question in a way that cannot be read off the prompt, such as
   how many customers are active, and see whether the agent goes back to the
   database.
2. Ask for the total revenue of the films a named actor appears in, and check
   the answer against a query you write yourself. The `film_actor` table has
   5462 rows for 1000 films, so a join through it repeats a payment once per
   actor.
3. Raise the temperature from 0 to 1 and ask the robot question five times.
   Count the distinct trajectories and the distinct answers, and say which of
   the two varies more.
4. Give `run_sql` a row limit of 3 instead of 20 and rerun Task 4. Work out
   which questions that breaks and why the score does not fall as far as you
   expect.